# ML-10 ? Content Action Playbook

## 1. Ranked actions + reason codes

This is a decision-support queue, not automated publishing. It uses out-of-fold client predictions so each score is generated by a model that did not train on that client. The primary action is a human `REVIEW_FOR_REFRESH`; a reason code explains the observable prior-window condition.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn pandas numpy
import os, duckdb, numpy as np, pandas as pd, json
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, average_precision_score
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert HF_TOKEN, "Add HF_TOKEN to Colab Secrets or the HF_TOKEN environment variable first."
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
pages = con.sql(f"""
SELECT client_hash_id, content_hash_id,
 SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impressions_prev,
 SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_clicks ELSE 0 END) AS clicks_prev,
 AVG(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_avg_position END) AS avg_position_prev,
 STDDEV(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_avg_position END) AS position_volatility_prev,
 COUNT(CASE WHEN report_date < DATE '2026-03-16' AND gsc_impressions > 0 THEN 1 END) AS active_days_prev,
 SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impressions_outcome
FROM {FACT} GROUP BY 1,2 HAVING impressions_prev >= 100
""").df()
pages["is_declining"] = (pages.impressions_outcome < 0.8 * pages.impressions_prev).astype(int)
features=["impressions_prev","clicks_prev","avg_position_prev","position_volatility_prev","active_days_prev"]
pages["log_impressions_prev"]=np.log1p(pages.impressions_prev); pages["log_clicks_prev"]=np.log1p(pages.clicks_prev)
features=["log_impressions_prev","log_clicks_prev","avg_position_prev","position_volatility_prev","active_days_prev"]
def p_at_k(y,s,k):
 k=min(k,len(y)); return float(pd.DataFrame({"y":np.asarray(y),"s":np.asarray(s)}).nlargest(k,"s").y.mean())
def baseline_score(d):
 return d.impressions_prev * (1 + (d.position_volatility_prev.fillna(0) >= 5).astype(int))
print(f"Page-level March frame: {len(pages):,} rows; positive rate: {pages.is_declining.mean():.1%}")
oof=np.zeros(len(pages)); cv=GroupKFold(n_splits=5)
for tr,te in cv.split(pages[features],pages.is_declining,groups=pages.client_hash_id):
 m=Pipeline([("impute",SimpleImputer(strategy="median")),("model",RandomForestClassifier(n_estimators=300,min_samples_leaf=25,class_weight="balanced_subsample",random_state=42,n_jobs=-1))])
 m.fit(pages.iloc[tr][features],pages.iloc[tr].is_declining); oof[te]=m.predict_proba(pages.iloc[te][features])[:,1]
queue=pages[["client_hash_id","content_hash_id","impressions_prev","avg_position_prev","position_volatility_prev"]].copy(); queue["review_score"]=oof
visible=queue.impressions_prev>=500; unstable=queue.position_volatility_prev.fillna(0)>=5
queue["reason_code"]=np.select([visible&unstable,visible],["visible_position_unstable","visible_search_history"],default="measurable_decline_risk")
queue["action_label"]=np.where(queue.review_score>=.5,"REVIEW_FOR_REFRESH","MONITOR")
queue["confidence"]=pd.cut(queue.review_score,[-.01,.5,.7,1.01],labels=["monitor","review","high_review"])
queue=queue.sort_values("review_score",ascending=False).reset_index(drop=True)
display(queue.head(20))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level March frame: 77,540 rows; positive rate: 28.5%


,client_hash_id,content_hash_id,impressions_prev,avg_position_prev,position_volatility_prev,review_score,reason_code,action_label,confidence
0,client_20259bd6705d81d4,content_a11bd5663919f057,30982.0,40.837551,2.004680,0.934249,visible_search_history,REVIEW_FOR_REFRESH,high_review
1,client_20259bd6705d81d4,content_f5e2cda099b4321c,9623.0,39.998754,2.497540,0.929717,visible_search_history,REVIEW_FOR_REFRESH,high_review
2,client_20259bd6705d81d4,content_c48f130f7bbaf8c5,11615.0,39.985984,2.626366,0.922431,visible_search_history,REVIEW_FOR_REFRESH,high_review
3,client_20259bd6705d81d4,content_f5b1433ef3cce601,8570.0,38.982037,2.692226,0.919177,visible_search_history,REVIEW_FOR_REFRESH,high_review
4,client_20259bd6705d81d4,content_83a700e06cf9e676,34003.0,42.743691,2.075525,0.916409,visible_search_history,REVIEW_FOR_REFRESH,high_review
5,client_fef1a8f436438636,content_0aaa197051f58d6f,26690.0,38.492268,2.698273,0.913445,visible_search_history,REVIEW_FOR_REFRESH,high_review
6,client_20259bd6705d81d4,content_599db0b571da0445,11942.0,34.529372,2.040361,0.913229,visible_search_history,REVIEW_FOR_REFRESH,high_review
7,client_20259bd6705d81d4,content_c7da032c307b766a,12907.0,35.870076,3.174232,0.912815,visible_search_history,REVIEW_FOR_REFRESH,high_review
8,client_fef1a8f436438636,content_bfd481a760fa3064,7386.0,42.510206,1.838741,0.911172,visible_search_history,REVIEW_FOR_REFRESH,high_review
9,client_20259bd6705d81d4,content_fbc4a78d5b8b8f0c,21897.0,37.012150,2.038646,0.910031,visible_search_history,REVIEW_FOR_REFRESH,high_review


## 2. Intended use and limits

A content/SEO reviewer uses the top of this queue to choose what to inspect first. Scores are comparable only within this March development population and this proxy definition. They are not a quality score, a Google ranking prediction, or proof that any edit will produce recovery.

In [2]:
print("Top-20 out-of-fold Precision@20 against this short-window proxy:",f"{p_at_k(pages.is_declining,oof,20):.1%}")
print("Population:",f"{len(queue):,} pages with at least 100 prior-half impressions.")


Top-20 out-of-fold Precision@20 against this short-window proxy: 25.0%
Population: 77,540 pages with at least 100 prior-half impressions.


## 3. Human review + the no-go list

Before acting, check the page?s current search context, intent, accuracy, technical accessibility, and whether the movement is temporary. Never automate deletion, content replacement, title changes, or claims about client impact from this queue. Do not use it as a substitute for editorial judgment or publish pseudonymous row-level exports outside the approved workflow.

In [3]:
no_go=["Automatic page deletion or publication","Treating a score as proof of a causal refresh effect","Using client/content identifiers as model inputs","Using later-half outcome metrics to score a new decision"]
pd.DataFrame({"no_go":no_go})


,no_go
0,Automatic page deletion or publication
1,Treating a score as proof of a causal refresh ...
2,Using client/content identifiers as model inputs
3,Using later-half outcome metrics to score a ne...


## 4. Monitoring / retrain triggers

Re-run when the reporting window changes. Investigate before reuse if the positive rate, Precision@K, missing-position rate, or distribution of prior impressions/position volatility shifts materially; retrain only after rechecking the feature/label timeline and client grouping.

In [4]:
monitor=pd.DataFrame({"metric":["decline-proxy base rate","OOF Precision@20","missing prior position share","median prior impressions"],"value":[pages.is_declining.mean(),p_at_k(pages.is_declining,oof,20),pages.avg_position_prev.isna().mean(),pages.impressions_prev.median()]})
monitor


,metric,value
0,decline-proxy base rate,0.285478
1,OOF Precision@20,0.250000
2,missing prior position share,0.000000
3,median prior impressions,576.000000


## 5. Exports for the paper

Only the pseudonymized review queue and aggregate metric receipt are written. The CSV remains in `work/outputs/` and is not a dataset to commit.

In [5]:
out=Path("work/outputs"); out.mkdir(parents=True,exist_ok=True)
queue[["client_hash_id","content_hash_id","review_score","reason_code","action_label","confidence"]].to_csv(out/"w07_action_playbook_queue.csv",index=False)
Path(out/"w07_action_playbook_metrics.json").write_text(json.dumps({"rows":int(len(queue)),"oof_precision_at_20":p_at_k(pages.is_declining,oof,20),"reason_code_counts":queue.reason_code.value_counts().to_dict()},indent=2))
print("Wrote work/outputs/w07_action_playbook_queue.csv and safe aggregate metrics JSON.")


Wrote work/outputs/w07_action_playbook_queue.csv and safe aggregate metrics JSON.


## Self-check

- [x] The queue is ranked by out-of-fold client scores
- [x] Every action has an observable reason code
- [x] Human checks and no-go actions are explicit
- [x] Monitoring triggers are defined